# Levels of Poker: Modeling Decision-Making Across Player Skill Tiers


Levels of poker decision-making — from a player who sees only their own hand to one who reads the full board, position, and betting story.

All preprocessing lives in `scripts/build_features.py`. Run it once before this notebook to produce the eight intermediate dataframes under `data/processed/`. This notebook just loads them and trains models on top.

## Setup

In [ ]:
!pip install pandas numpy scikit-learn matplotlib xgboost pyarrow

import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb

warnings.filterwarnings('ignore')

PROCESSED = 'data/processed'
print('Setup complete.')

First time? Run the preprocessing script before going further. It produces every dataframe loaded below.

On Colab, mount Drive (or upload the raw CSV) and then:
```
!python -m scripts.build_features path/to/postflop_500k_train_set_game_scenario_information.csv
```

In [ ]:
def plot_importances(clf, feature_cols, title):
    indices = np.argsort(clf.feature_importances_)[::-1]
    sorted_features = [feature_cols[i] for i in indices]
    sorted_importances = clf.feature_importances_[indices]
    plt.figure(figsize=(10, max(3, 0.4 * len(feature_cols))))
    plt.barh(sorted_features[::-1], sorted_importances[::-1],
             color='steelblue', edgecolor='black')
    plt.xlabel('Importance'); plt.ylabel('Feature')
    plt.title(title); plt.tight_layout(); plt.show()

def show_tree(clf, X):
    plt.figure(figsize=(20, 10))
    plot_tree(clf, feature_names=X.columns, class_names=clf.classes_,
              filled=True, rounded=True, fontsize=8)
    plt.show()

def fit_and_report(df_features, feature_cols, target='decision_category',
                   max_depth=5, title=''):
    """Train a depth-`max_depth` tree, print accuracies, plot importances + tree."""
    X = df_features[feature_cols]
    y = df_features[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    clf = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    clf.fit(X_train, y_train)
    print(f'Train accuracy: {accuracy_score(y_train, clf.predict(X_train)):.4f}')
    print(f'Test accuracy:  {accuracy_score(y_test, clf.predict(X_test)):.4f}')
    plot_importances(clf, feature_cols, f'{title}: feature importances')
    show_tree(clf, X)
    return clf

## Introduction

- Poker Basics
- Streets, actions, antes
- GTO

## Motivation

- Increase popularity of non-professionals taking a shot at the main event
- Combines skill / luck -> is this a gambling epidemic?
- Want to examine the "levels" of players and learnability of the game

In [ ]:
import os, kagglehub

wsop_path = kagglehub.dataset_download(
    'cviaxmiwnptr/wsop-main-event-results-1971-2024'
)
csv_file = [f for f in os.listdir(wsop_path) if f.endswith('.csv')][0]
wsop = pd.read_csv(os.path.join(wsop_path, csv_file))
entries_by_year = wsop.groupby('year')['entries'].first().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(entries_by_year['year'], entries_by_year['entries'],
        marker='o', linewidth=2, color='#c41e3a')
ax.fill_between(entries_by_year['year'], entries_by_year['entries'],
                alpha=0.15, color='#c41e3a')
ax.set_xlabel('Year'); ax.set_ylabel('Number of entries')
ax.set_title('WSOP Main Event entries by year (1971–2024)',
             fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_xticks(entries_by_year['year'].values[::5])
plt.xticks(rotation=45)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

## Data Loading

- PokerBench postflop dataset (500k GTO-labeled hands)
- Each row = one decision point: hole cards, board so far, preflop + postflop action, available moves, GTO-correct decision
- Cleaning + decision-category simplification done in `scripts/build_features.py`

In [ ]:
df_simplified = pd.read_parquet(f'{PROCESSED}/df_simplified.parquet')
print(f'{len(df_simplified):,} rows')
df_simplified.head()

## Baseline

- Random-guess baseline: pick uniformly among available moves
- Compute both analytical expectation and simulated average
- Based on PokerBench research paper:
- Pre-trained ChatGPT-4 achieved action accuracy of 62.69%
- Fine-tuned LLAMA-3 achieved action accuracy of 79.07%

In [ ]:
moves = df_simplified['available_moves_simplified'].str.split(',')
expected_accuracy = (1 / moves.apply(len)).mean()
print(f'Expected random-baseline accuracy: {expected_accuracy:.4f} '
      f'({expected_accuracy*100:.2f}%)')

print('\nAvailable-move counts:')
print(moves.apply(len).value_counts().sort_index())

In [ ]:
rng = np.random.default_rng(42)
n_runs = 10
accuracies = []
for _ in range(n_runs):
    guesses = moves.apply(lambda m: rng.choice(m))
    accuracies.append((guesses == df_simplified['decision_category']).mean())
print(f'Simulated random accuracy ({n_runs} runs): '
      f'{np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}')

## Level 1 — Basic Player (Own Hand Only)

- Sees only own two hole cards + available moves
- Holding features: rank_sum, rank_max, is_pair
- Move-availability flags: facing_bet, can_bet_raise

In [ ]:
df_own_hand = pd.read_parquet(f'{PROCESSED}/df_own_hand.parquet')
df_own_hand.head()

In [ ]:
feature_cols = ['is_pair', 'rank_sum', 'rank_max', 'can_bet_raise', 'facing_bet']
fit_and_report(df_own_hand, feature_cols, title='Level 1')

## Level 2 — Hand Strength

- Add 5-card hand evaluation (hole + visible board)
- Convert hand-strength tuples to a single ordinal `hand_rank`
- Question: does knowing the strength of the made hand help?

In [ ]:
df_hand_strength_final = pd.read_parquet(
    f'{PROCESSED}/df_hand_strength_final.parquet'
)
df_hand_strength_final.head()

In [ ]:
# Drop the human-readable category before training (we use the ordinal rank)
df_l2 = df_hand_strength_final.drop(columns=['hand_category'])
feature_cols = ['is_pair', 'rank_sum', 'rank_max', 'can_bet_raise',
                'facing_bet', 'hand_rank', 'street']
fit_and_report(df_l2, feature_cols, title='Level 2')

## Level 3 — Max Future Strength

- Add max-future strength: best hand reachable by the river
- Walk hand-rank ladder top-down, stop at first reachable category
  (royal flush -> straight flush -> quads -> ...)
- Co-rank current and future tuples on a shared scale

In [ ]:
df_max_strength_final = pd.read_parquet(
    f'{PROCESSED}/df_max_strength_final.parquet'
)
df_max_strength_final.head()

In [ ]:
df_l3 = df_max_strength_final.drop(columns=['hand_category'])
feature_cols = ['is_pair', 'rank_sum', 'rank_max', 'can_bet_raise',
                'facing_bet', 'hand_rank', 'max_future_rank', 'street']
fit_and_report(df_l3, feature_cols, title='Level 3')

## Level 4 — Board Strength (Opponent's Nut)

- Add `opponent_nut_rank`: best hand any opponent could plausibly hold
- Accounts for player's hole cards as blockers
- Accounts for any future board cards still to come

In [ ]:
df_nut_strength_final = pd.read_parquet(
    f'{PROCESSED}/df_nut_strength_final.parquet'
)
df_nut_strength_final.head()

In [ ]:
df_l4 = df_nut_strength_final.drop(columns=['hand_category'])
feature_cols = ['is_pair', 'rank_sum', 'rank_max', 'can_bet_raise',
                'facing_bet', 'hand_rank', 'max_future_rank', 'street',
                'opponent_nut_rank']
fit_and_report(df_l4, feature_cols, title='Level 4')

## Level 5 — Basic Context (Position + Pot)

- Add position (in-position vs out-of-position)
- Add pot size
- Add aggressor's position

In [ ]:
df_context = pd.read_parquet(f'{PROCESSED}/df_context.parquet')
df_context.head()

In [ ]:
df_l5 = df_context.drop(columns=['hand_category'])
feature_cols = ['is_pair', 'rank_sum', 'rank_max', 'can_bet_raise',
                'facing_bet', 'hand_rank', 'max_future_rank', 'street',
                'opponent_nut_rank', 'pot_size', 'hero_position',
                'aggressor_position']
fit_and_report(df_l5, feature_cols, title='Level 5')

## Level 6 — Action Sequences

- Add preflop summary: num raises, num players, last raise size (bb)
- Add postflop summary: bets/raises on current street, last bet size as % pot

In [ ]:
df_action = pd.read_parquet(f'{PROCESSED}/df_action.parquet')
df_action.head()

In [ ]:
df_l6 = df_action.drop(columns=['hand_category'])
feature_cols = ['is_pair', 'rank_sum', 'rank_max', 'can_bet_raise',
                'facing_bet', 'hand_rank', 'max_future_rank', 'street',
                'opponent_nut_rank', 'pot_size', 'hero_position',
                'aggressor_position', 'pre_num_raises', 'pre_num_players',
                'pre_last_raise_bb', 'post_bets_raises',
                'post_last_bet_pct_pot']
fit_and_report(df_l6, feature_cols, title='Level 6')

## Exploring Tree Depth

- Sweep `max_depth` from 1 to 20 with the full feature set
- Plot train vs test accuracy to see bias-variance trade-off
- Pick the depth with the best held-out accuracy

In [ ]:
df_full = df_action.drop(columns=['hand_category'])
feature_cols = [c for c in df_full.columns if c != 'decision_category']
X = df_full[feature_cols]
y = df_full['decision_category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

depths = list(range(1, 21))
train_accs, test_accs = [], []
for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, clf.predict(X_train)))
    test_accs.append(accuracy_score(y_test, clf.predict(X_test)))

plt.figure(figsize=(10, 6))
plt.plot(depths, train_accs, marker='o', label='Train', color='steelblue')
plt.plot(depths, test_accs, marker='o', label='Test', color='crimson')
plt.xlabel('Max depth'); plt.ylabel('Accuracy')
plt.title('Decision tree accuracy vs. depth')
plt.xticks(depths); plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()

best_idx = int(np.argmax(test_accs))
best_depth = depths[best_idx]
print(f'Best test accuracy: {test_accs[best_idx]:.4f} at depth {best_depth}')

In [ ]:
best_clf = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
best_clf.fit(X_train, y_train)

indices = np.argsort(best_clf.feature_importances_)[::-1]
sorted_features = [feature_cols[i] for i in indices]
sorted_importances = best_clf.feature_importances_[indices]

plt.figure(figsize=(12, 6))
plt.bar(sorted_features, sorted_importances, color='steelblue', edgecolor='black')
plt.title(f'Feature importances at best depth ({best_depth}, '
          f'test acc {test_accs[best_idx]:.4f})')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

for feat, imp in zip(sorted_features, sorted_importances):
    print(f'  {feat:30s} {imp:.4f}')

## XGBoost

- Same engineered feature set, stronger model
- 500 trees, depth 6, early stopping on validation log-loss

In [ ]:
feature_cols = [c for c in df_full.columns if c != 'decision_category']
X = df_full[feature_cols]
y = df_full['decision_category']

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

clf = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.1,
    objective='multi:softprob', eval_metric='mlogloss',
    early_stopping_rounds=20, random_state=42, n_jobs=-1,
    tree_method='hist',
)
clf.fit(X_train, y_train,
        eval_set=[(X_train, y_train), (X_test, y_test)], verbose=50)

print(f'\nTrain accuracy: {accuracy_score(y_train, clf.predict(X_train)):.4f}')
print(f'Test accuracy:  {accuracy_score(y_test, clf.predict(X_test)):.4f}')

fig, ax = plt.subplots(figsize=(10, 6))
xgb.plot_importance(clf, ax=ax, importance_type='gain', max_num_features=20)
plt.tight_layout(); plt.show()

## XGBoost with Natural Features

- Card one-hots for every hole + board slot (7 × 52 columns)
- Board aggregate features: suit counts, rank duplications, connectedness
- Per-street action sequence features: counts of bet/raise/check/call, max + sum bet sizes
- Lets XGBoost learn poker structure from rawer inputs

In [ ]:
df_rich = pd.read_parquet(f'{PROCESSED}/df_rich.parquet')
print(f'{df_rich.shape[1]} columns total ({df_rich.shape[1] - df_action.shape[1]} added)')
df_rich.head()

In [ ]:
df_rich_nocat = df_rich.drop(columns=['hand_category'], errors='ignore')
feature_cols = [c for c in df_rich_nocat.columns if c != 'decision_category']
X = df_rich_nocat[feature_cols]
y = df_rich_nocat['decision_category']

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

clf = xgb.XGBClassifier(
    n_estimators=1000, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective='multi:softprob', eval_metric='mlogloss',
    early_stopping_rounds=30, random_state=42, n_jobs=-1,
    tree_method='hist',
)
clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)

print(f'\nTest accuracy: {accuracy_score(y_test, clf.predict(X_test)):.4f}')

## Conclusions

- Each feature group adds something to the model
- Decision tree accuracy climbs steadily Levels 1 -> 6
- XGBoost on engineered features pushes the ceiling further
- XGBoost + natural features gives the model the most freedom
- Takeaway for learnability: own-hand-only beats random, but approaching GTO requires reasoning about board, opponents, position, and betting story together

## Export requirements.txt

In [ ]:
!pip freeze > requirements.txt
from google.colab import files
files.download('requirements.txt')
!python --version